# verl处理“训练”与“推理”底层引擎不同，导致的logprob 口径不一致

针对大模型强化学习（RL）中由于“训练”与“推理”底层引擎不同，导致的 logprob 口径不一致（即 Importance Ratio 计算失准）这一致命痛点，VeRL（Volcano Engine Reinforcement Learning）框架在设计上通过以下三个层面进行了系统性的处理和规避：

## 一、 核心策略：弃用推理侧概率，强制训练侧重算
<font color='red'>这是 VeRL 框架中最根本、最核心的解决方案</font>

在训练阶段，当拿到推理引擎（如 vLLM）生成的 rollout 序列后，<font color='red'>VeRL 绝对不会直接使用推理引擎生成时返回的 logprob 来作为分母</font>（即旧策略概率 $π_{old}$）计算 ratio。

- 处理方式：
  - VeRL 会在训练阶段（Training Stage）<font color='red'>将生成的文本喂回训练引擎</font>（如 FSDP / Megatron），重新前向传播（Recompute） 计算出一套标准化的、精确的 old_log_probs。
- 底层原因：
  - 推理引擎为了追求高吞吐，通常会走“快路径”（Fast Path），
    - 在底层计算 logits 时可能会引入温度（temperature）、top-p 截断、重复惩罚（repetition penalty）<font color='red'>等处理器改写 logit</font>，**甚至底层算子的累加顺序与训练侧不同**。
  - <font color='red'>推理侧返回的 logprob 是“被采样策略污染”的</font>，直接用于训练会导致梯度方向错误。**而训练侧的重算使用的是原始、干净的前向计算，能够保证分子和分母在同一套标准下比较。**


### 核心思路

rollout 只负责生成序列，训练阶段再用训练引擎重新前向计算 log_prob，并以此作为策略更新的依据

在 PPO / GRPO 这类 RLHF 流程里，Actor 有两个用途：
1. Rollout 阶段：用推理引擎生成回答。
2. Training 阶段：用训练引擎重算 log_prob、KL、advantage、loss。

VeRL 把这两个阶段拆开：
- <font color='red'>推理侧负责 generate_sequences</font>，
- 训练侧负责 compute_log_prob 和 update_actor。

### 训练循环伪代码

In [ ]:
for batch in dataloader:
    # 1. Rollout：只生成序列，不把推理侧 logprob 直接当训练依据
    rollout_batch = actor.generate_sequences(batch)

    # 2. 经验准备：在训练侧重算 log_prob、reward、value、advantage
    rollout_batch = reward.compute_reward(rollout_batch)
    rollout_batch = reference.compute_log_prob(rollout_batch)   # 可选：参考策略 KL
    rollout_batch = critic.compute_values(rollout_batch)        # PPO 需要
    rollout_batch = compute_advantage(rollout_batch)

    # 3. 强制训练侧重算：用训练引擎重新计算策略 log_prob
    rollout_batch = actor.compute_log_prob(rollout_batch)

    # 4. 策略更新：基于训练侧 log_prob 计算 loss
    actor.update_actor(rollout_batch)
    critic.update_critic(rollout_batch)

In [1]:
# ---------- 推理侧：只负责采样 ----------
class ActorRolloutEngine:
    def generate_sequences(self, batch):
        # 使用 vLLM / SGLang 等推理引擎生成 response
        responses = self.inference_engine.generate(
            prompts=batch["prompts"],
            max_length=self.max_response_length,
            temperature=self.temperature,
        )
        return DataProto(prompts=batch["prompts"], responses=responses)

# ---------- 训练侧：强制重算 log_prob ----------
class ActorTrainEngine:
    def compute_log_prob(self, batch):
        # 使用 FSDP / Megatron 等训练引擎重算
        # 这里不能用 rollout 引擎输出的 logprob 直接替代
        logits = self.train_model.forward(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        log_prob = compute_token_log_prob(logits, batch["labels"])
        batch["log_prob"] = log_prob
        return batch

    def update_actor(self, batch):
        log_prob = batch["log_prob"]
        old_log_prob = batch["old_log_prob"]
        advantage = batch["advantage"]

        ratio = torch.exp(log_prob - old_log_prob.detach())
        surr1 = ratio * advantage
        surr2 = torch.clamp(ratio, 1 - self.clip_range, 1 + self.clip_range) * advantage
        policy_loss = -torch.min(surr1, surr2).mean()

        loss = policy_loss
        loss.backward()
        self.optimizer.step()

## 二、 兜底策略：Bypass 模式（快速但有风险的捷径）
VeRL 允许用户通过配置开启一种名为 Bypass 模式 的捷径。
- 处理方式：
  - 在这种模式下，系统会跳过训练侧耗时的 logprob 重算步骤，直接用推理引擎返回的 rollout_log_probs 充当 old_log_probs 来计算 PPO ratio。
- 风险提示：
  - <font color='red'>这本质上是一种用“数值精度”换“训练速度”的妥协</font>。
    - <font color='blue'>只有当你确认推理引擎和训练引擎在底层数值上做到了高度一致（例如 tokenizer 完全对齐、无特殊采样器干扰、序列长度适中且模型较小）时，才建议使用。</font>对于 DeepSeek-V3 等长 CoT（思维链）或训推并行度差异大的场景，极易导致 ratio 方差失控。



## 三、 容错策略：Rollout Correction（训推失配修正）
对于那些为了极致性能（如 Fully Async 全异步训练）不得不接受训推参数不同步或底层计算存在固有微小差异的场景，VeRL 提供了 Rollout Correction 机制。

- 处理方式：
  - 它将 PPO 的 Loss 计算进行了数学上的解耦。它并不强行要求 $π_{old}$ 绝对等于 rollout 时的行为策略概率，而是将 rollout 阶段的策略（ $π_{rollout}$）和 training 阶段的策略（ $π_{old}$）<font color='red'>分开处理</font>
- 底层逻辑：
  - PPO 传统的 Clip 约束负责限制“本轮 optimizer 更新步长”；
  - <font color='red'>而对于训推不一致带来的偏差，框架可以通过额外的 IS（重要性采样）加权、截断（truncation）或拒绝掩码（rejection mask）来进行修正补偿，从而允许一定程度的训推底层误差存在，保证训练流程的稳定性。</font>

# compute_log_prob实现

在 VeRL 这类 RLHF 框架里，本质上是一个“训练侧重算接口”：
  - 它把 rollout 阶段生成的 prompt + response 拼成完整序列，重新走一遍训练模型的前向传播，然后只取 response 部分的 token log probability。
  - 它的关键不是“算出概率”，而是：
    - <font color='red'>保证训练侧 log_prob 与 rollout 序列、loss mask、padding mask 完全对齐。</font>

## 最简实现

In [ ]:
import torch
import torch.nn.functional as F

def compute_log_prob(
    model,
    input_ids,
    attention_mask=None,
    response_length=None,
    loss_mask=None,
    pad_token_id=0,
):
    """
    训练侧重算策略 log_prob。

    Args:
        input_ids: [B, L]，prompt + response 的完整序列
        attention_mask: [B, L]，可选
        response_length: int 或 [B]，response 部分长度
        loss_mask: [B, L]，可选，若提供则优先使用
        pad_token_id: padding token id = 0 

    Returns:
        log_prob: [B, L_resp]，只覆盖 response 部分
        loss_mask: [B, L_resp]
        
    input_ids = torch.tensor([
         [101, 2054, 3421,  102,    0],   # 第 0 条样本
         [101, 4532,  876, 1034,    0],   # 第 1 条样本
         [101, 7821,  102,    0,    0],   # 第 2 条样本
    ])    
        
    """
    B, L = input_ids.shape

    if attention_mask is None:
        """
        long() 的作用是把张量转成 torch.int64（64 位整数）类型。
        """
        attention_mask = (input_ids != pad_token_id).long()

    # 前向传播，得到每个位置的 logits
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits  # [B, L, V]

    # 构造 shift：t 时刻预测 t+1
    """
    这两行代码的本质是：把模型输出的预测值（logits）和真实的下一个词（labels）在时间轴上对齐
    - 不 Shift 的话，模型会用“预测 C 的 logits”去训练“标签 B”，这就完全乱套了
    
    含义：取所有样本（:），抛弃最后一个 token 位置的预测结果EOS（:-1），保留所有词表的预测值（:）
    """
    shift_logits = logits[:, :-1, :].contiguous()   # [B, L-1, V]
    """
    1: 去掉第一个词，把序列左移一位
    
    此时shift_labels 就是我们希望模型预测的”下一个词“
    """
    shift_labels = input_ids[:, 1:].contiguous()    # [B, L-1]

    # 每个位置的 log_prob
    token_log_prob = -F.cross_entropy(
        shift_logits.reshape(-1, shift_logits.size(-1)),
        shift_labels.reshape(-1),
        reduction="none",
    ).reshape(B, L - 1)  # [B, L-1]

    # 构造 response 部分 mask
    if loss_mask is None:
        assert response_length is not None, "需要提供 response_length 或 loss_mask"
        if isinstance(response_length, int):
            resp_start = L - response_length
            loss_mask = torch.zeros_like(shift_labels, dtype=torch.bool)
            loss_mask[:, resp_start - 1:] = True  # 对齐 shift 后的位置
        else:
            # response_length: [B]
            resp_starts = L - response_length  # [B]
            indices = torch.arange(L - 1, device=input_ids.device).unsqueeze(0)  # [1, L-1]
            loss_mask = indices >= (resp_starts - 1).unsqueeze(1)

    # 屏蔽 padding / prompt 部分
    token_log_prob = token_log_prob * loss_mask.float()

    return token_log_prob, loss_mask

## 逐层解释
### 为什么必须重走前向？
1. rollout 阶段通常使用 vLLM、SGLang 等推理引擎，它们为了吞吐会做：
  - KV cache；
  - 采样调度；
  - 可能不同的 precision；
  - 可能不同的 tokenizer 或 padding 策略；
  - 输出的 logprob 只覆盖采样 token，不一定覆盖完整 response。
2. 如果直接拿推理侧 logprob 进 loss，容易出现：
  - 训练模型和推理模型状态不一致；
  - token 对齐出错；
  - padding 被错误计入 loss；
  - prompt 部分被错误计入 loss；
  - clip ratio 计算不稳定。
所以 VeRL 的设计是：rollout 只负责生成序列，训练侧用同一套训练模型重新算 log_prob。

### <font color='red'>shift 的作用</font>
语言模型训练目标是：
  - $logp(x_t ∣x_{<t})$
  
但模型输出 logits[t] 对应的是预测 x[t+1]，所以需要 shift：

In [ ]:
shift_logits = logits[:, :-1]   # 预测下一个 token
shift_labels = input_ids[:, 1:] # 真正的下一个 token

也就是说：

| 位置 | logits 来源 | 预测目标 |
|---|---|---|
| `logits[0]` | `x[0]` | `x[1]` |
| `logits[1]` | `x[0], x[1]` | `x[2]` |
| `logits[t]` | `x[:t+1]` | `x[t+1]` |

### 为什么只取 response 部分？
- RLHF 里通常<font color='red'>不对 prompt 做策略更新</font>，因为 prompt 是环境给定的，不是策略生成的。
- <font color='blue'>所以 compute_log_prob 必须把 loss mask 限制在 response 区间：</font>

In [ ]:
# 假设完整序列是 [prompt, response]
# response_length = L_resp
# 那么 response 在 input_ids 里的起始位置是 L - L_resp

<font color='red'>注意 shift 会让长度减 1，所以 mask 起始位置要减 1。</font>

###  loss_mask 的几种来源
实际工程里，loss_mask 可能来自：
  - rollout 阶段返回的 response mask；
  - tokenizer 生成的 attention_mask；
  - 根据 response_length 推导；
  - 根据特殊 token，比如 <response_start>、<response_end> 定位；
  - 多轮对话里只 mask assistant 回复部分。
更鲁棒的写法是：

In [ ]:
if loss_mask is not None:
    # 优先使用外部提供的 mask
    return apply_mask(token_log_prob, loss_mask)
else:
    # 退而使用 response_length 推导
    return derive_mask_from_response_length(token_log_prob, response_length)

## 带 VeRL 风格的封装

In [ ]:
class ActorComputeLogProb:
    def __init__(self, model, pad_token_id=0):
        self.model = model
        self.pad_token_id = pad_token_id

    def __call__(self, batch):
        input_ids = batch["input_ids"]
        attention_mask = batch.get("attention_mask")
        loss_mask = batch.get("loss_mask")
        response_length = batch.get("response_length")

        log_prob, mask = compute_log_prob(
            model=self.model,
            input_ids=input_ids,
            attention_mask=attention_mask,
            response_length=response_length,
            loss_mask=loss_mask,
            pad_token_id=self.pad_token_id,
        )

        batch["log_prob"] = log_prob
        batch["loss_mask"] = mask
        return batch

### 调用方式类似：

In [ ]:
batch = actor_compute_log_prob(batch)
log_prob = batch["log_prob"]

## 与 PPO / GRPO 的关系
在 PPO 里，compute_log_prob 通常会被调用两次：

In [ ]:
# 第一次：用当前策略重算，作为 old_log_prob
batch = actor.compute_log_prob(batch)
batch["old_log_prob"] = batch["log_prob"].detach()

# 第二次：在 update 阶段再次重算，作为当前策略 log_prob
batch = actor.compute_log_prob(batch)
current_log_prob = batch["log_prob"]

然后计算 ratio：

In [ ]:
ratio = torch.exp(current_log_prob - old_log_prob)

在 GRPO 里，如果没有 critic，优势通常来自组内 reward 归一化，但策略部分仍然需要训练侧 log_prob。

## 工程上最容易踩的坑
  - tokenizer 不一致：
    - rollout 和 train 必须使用相同 tokenizer。
  - padding 方向不一致：
    - left padding 和 right padding 会改变 response 位置。
  - response_length 计算错误：
    - 多算或少算一个 token，mask 就会错位。
  - loss_mask 没对齐 shift：
    - shift 后长度是 L-1，mask 也要对应。
  - 推理侧 logprob 混入训练：
    - 必须明确只把训练侧 compute_log_prob 的结果送进 loss。
  - 多轮对话 mask 错误：
    - 如果只训练 assistant 回复，需要精确 mask user/system 部分。
  
所以，compute_log_prob 的核心实现可以概括成一句话：
  - <font color='red'>用训练模型对完整序列做一次前向，构造 shift 交叉熵，得到每个 token 的 log probability</font>
  - 再通过 loss_mask 只保留 response 部分，从而保证策略更新完全依赖训练侧重算概率。
